**딥러닝 훈련을 안정화하고 성능을 높이는 대표적 기술**



# 배치 정규화(Batch Normalization)

Gradient Vanishing/Gradient Exploding이 일어나는 문제를 방지하기 위해서 사용

학습에 불안정화가 일어나는 이유 <- internal Covariance Shift (각 레이어나 Activation마다 입력값의 분산이 달라지는 현상)

이런 현상을 막기 위해서 각 레이어의 입력의 분산을 평균 0, 표준편차 1인 입력값으로 정규화하는 방법을 생각할 수 있음(Whitening)

**Whitening 문제점**
계산량이 많고 일부 파라미터들의 영향이 무시 -> 특정 파라미터가 계속 커지는 상태가 될 수 있음


배치 정규화 : 각 레이어마다 정규화 하는 레이어를 두어, 변형된 분포가 나오지 않도록 조절하게 하는 것

장점: 과적합 감소, 더 큰 학습률 사용 가능




In [ ]:
#완전연결(MLP)에서 BatchNormalization 쓰기
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(784,)),

    layers.Dense(256),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.Dense(128),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.Dense(10, activation='softmax')
])


#학습률 스케줄링(Learning Rate Scheduling)

가중치를 최적화하는데 있어서 좋은 학습률을 찾는 것이 중요함

학습률은 걸음의 보폭이기 때문에 학습률이 너무 크면 최적의 학습률을 지나치거나 학습률이 너무 작아도 최적의 학습률을 찾을 수 있지만 시간이 매우 오래 걸릴 것임


학습률 스케쥴링의 대표 방식
- step Decay :일정 에폭마다 LR을 갑자기 낮춤
- Exponential Decay : 에폭이 증가할 수록 LR을 지수적으로 감소
- Cosine Annealing : 코사인 곡선처럼 감소
- Warmup : 초반에 LR을 작게 시작 점점 크게
- RedeceLROnPlateau : 성능 개선 멈추면 LR 감소

학습 초기에 너무 작으면 느리고, 너무 크면 발산

In [ ]:
from tensorflow.keras.callbacks import LearningRateScheduler, ReduceLROnPlateau
import math

In [ ]:
def step_decay(epoch):
    return 0.001 * (0.5 ** (epoch // 10))

callback = LearningRateScheduler(step_decay)

In [ ]:
def exp_decay(epoch):
    return 0.001 * math.exp(-0.1 * epoch)

callback = LearningRateScheduler(exp_decay)

In [ ]:
def cosine_annealing(epoch, max_epochs=50):
    return 0.001 * (1 + math.cos(math.pi * epoch / max_epochs)) / 2

callback = LearningRateScheduler(cosine_annealing)


In [ ]:
def warmup(epoch):
    if epoch < 5:
        return 0.001 * (epoch + 1) / 5
    return 0.001

callback = LearningRateScheduler(warmup)


In [ ]:
callback = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

# 데이터 증강(Data Augmentation)

모델을 훈련시킬 때, 훈련 데이터가 적은 경우에 사용

훈련 데이터가 적으면?

- 훈련 데이터 세트에 대해 과적합 -> 훈련 세트에 대해 과하게 학습
- 불균형 데이터 -> 특정 클래스에 대한 훈련 데이터가 적을 때 발생


**이미지 데이터 증강**

이미지에 약간 변형을 주는 방법

- 랜덤하게 이미지 자르기
- 회전
- 밝기 조절
- 블러 처리
- 노이즈 삽입

데이터 분포를 왜곡할 위험이 있음
  

In [ ]:
#이미지 폴더에서 불러오면서 증
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

train_flow = train_gen.flow_from_directory(
    'data/train/',
    target_size=(224, 224),
    batch_size=32
)

model.fit(train_flow, epochs=10) #학습

In [ ]:
#모델 앞단에 레이어로 추가하는 형태

from tensorflow.keras import layers, Sequential

data_augmentation = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])


In [ ]:
model = Sequential([
    data_augmentation,
    layers.Conv2D(32, 3, activation='relu'),
    ...
])


# 가중치 초기화

가중치가 너무 작거나 너무 크면 학습 과정에서 vanishing gradient/exploding gradient 문제가 발생할 수 있음

**가중치 초기화 대표 방식**
- Xavier
- He

**Xavier 초기화**

각 층 사이의 가중치들에 연결된 입력과 출력 노드의 개수에 맞게 가중치를 랜덤하게 설정하는 방식

이 방식은 하이퍼볼릭 탄젠트나 시그모이드 활성화 함수와 잘 어울림

균등분포와 정규분포 두 종류로 나눔

**He 초기화**

ReLU와 같은 활성화 함수에 적합하며
분산이 2/fan_in으로 설정됨

균등분포와 정규분포 두 종류로 나눔

In [ ]:
layers.Dense(128, activation='relu',
             kernel_initializer='he_normal')

In [ ]:
layers.Dense(128, activation='tanh',
             kernel_initializer='glorot_uniform')
